In [0]:
path = "/Volumes/workspace/default/raw_eurostat/estat_ilc_pw01.tsv"

raw = (spark.read
       .option("header", True)
       .option("sep", "\t")        # the columns are separated by tabs
       .option("inferSchema", False)  # keep everything as plain text, untouched
       .csv(path))

raw.printSchema()
print("rows:", raw.count())
display(raw.limit(5))

root
 |-- freq,statinfo,unit,isced11,life_sat,sex,age,geo\TIME_PERIOD: string (nullable = true)
 |-- 2013 : string (nullable = true)
 |-- 2018 : string (nullable = true)
 |-- 2021 : string (nullable = true)
 |-- 2022 : string (nullable = true)
 |-- 2023 : string (nullable = true)
 |-- 2024 : string (nullable = true)
 |-- 2025 : string (nullable = true)

rows: 6240


"freq,statinfo,unit,isced11,life_sat,sex,age,geo\TIME_PERIOD",2013,2018,2021,2022,2023,2024,2025
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,AL",:,6.3,6.3,6.6,6.9,6.8,7.3
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,AT",8.4,8.3,8.2,8.1,7.9,7.7,8.0
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,BE",7.7,7.9,7.5,7.8,7.8,7.9,7.9
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,BG",6.2,6.7,6.8,6.4,6.9,7.2,7.5
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,CH",8.3,8.5,7.9,7.5,6.8,7.2,:


In [0]:
cols = raw.columns
new_cols = ["series_key"] + ["y" + c.strip() for c in cols[1:]]   # 2013 -> y2013

bronze = raw.toDF(*new_cols)
bronze.write.mode("overwrite").saveAsTable("workspace.default.bronze_life_satisfaction")

check = spark.table("workspace.default.bronze_life_satisfaction")
print("saved rows:", check.count())
print(check.columns)
display(check.limit(5))

saved rows: 6240
['series_key', 'y2013', 'y2018', 'y2021', 'y2022', 'y2023', 'y2024', 'y2025']


series_key,y2013,y2018,y2021,y2022,y2023,y2024,y2025
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,AL",:,6.3,6.3,6.6,6.9,6.8,7.3
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,AT",8.4,8.3,8.2,8.1,7.9,7.7,8.0
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,BE",7.7,7.9,7.5,7.8,7.8,7.9,7.9
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,BG",6.2,6.7,6.8,6.4,6.9,7.2,7.5
"A,AVG,RTG,ED0-2,LIFE,F,Y16-19,CH",8.3,8.5,7.9,7.5,6.8,7.2,:


In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.default.bronze_life_satisfaction")
names = ["freq","statinfo","unit","isced11","life_sat","sex","age","geo"]
year_cols = [c for c in bronze.columns if c.startswith("y")]

# 1. Unglue the first column into 8 columns
parts = F.split(F.col("series_key"), ",")
unjammed = bronze.select(
    *[parts.getItem(i).alias(n) for i, n in enumerate(names)],
    *year_cols
)

# 2. Rotate: year columns become rows
pairs = ", ".join([f"'{c[1:]}', `{c}`" for c in year_cols])
long_df = unjammed.selectExpr(*names, f"stack({len(year_cols)}, {pairs}) as (year, raw_value)")

# 3. Separate the number from the sticker, and flag problems
number_pattern = r"^([0-9]+(\.[0-9]+)?)"
clean = (long_df
    .withColumn("raw_value", F.trim("raw_value"))
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("is_missing",
        F.col("raw_value").isNull() | F.col("raw_value").startswith(":") | (F.col("raw_value") == ""))   # CHANGED
    .withColumn("value",
        F.when(F.col("raw_value").rlike(number_pattern),
               F.regexp_extract("raw_value", number_pattern, 1).cast("double")))
    .withColumn("flag",
        F.when(F.col("raw_value").rlike(r"^[0-9.:]+\s*[A-Za-z]+$"),
               F.regexp_extract("raw_value", r"([A-Za-z]+)$", 1)))   # CHANGED
    .withColumn("is_out_of_range",
        F.col("value").isNotNull() & ((F.col("value") < 0) | (F.col("value") > 10)))
)

print("rows after rotating:", clean.count())
display(clean.limit(8))

rows after rotating: 43680


freq,statinfo,unit,isced11,life_sat,sex,age,geo,year,raw_value,is_missing,value,flag,is_out_of_range
A,AVG,RTG,ED0-2,LIFE,F,Y16-19,AL,2013,:,true,null,null,false
A,AVG,RTG,ED0-2,LIFE,F,Y16-19,AT,2013,8.4,false,8.4,null,false
A,AVG,RTG,ED0-2,LIFE,F,Y16-19,BE,2013,7.7,false,7.7,null,false
A,AVG,RTG,ED0-2,LIFE,F,Y16-19,BG,2013,6.2,false,6.2,null,false
A,AVG,RTG,ED0-2,LIFE,F,Y16-19,CH,2013,8.3,false,8.3,null,false
A,AVG,RTG,ED0-2,LIFE,F,Y16-19,CY,2013,7.6,false,7.6,null,false
A,AVG,RTG,ED0-2,LIFE,F,Y16-19,CZ,2013,8.0 u,false,8.0,u,false
A,AVG,RTG,ED0-2,LIFE,F,Y16-19,DE,2013,7.6,false,7.6,null,false


In [0]:
rows_before = clean.count()

silver = (clean
    .dropDuplicates(names + ["year"])
    .select(*names, "year", "raw_value", "value", "flag", "is_missing", "is_out_of_range"))

rows_after = silver.count()

silver.write.mode("overwrite").saveAsTable("workspace.default.silver_life_satisfaction")

audit = spark.createDataFrame(
    [(rows_before, rows_after, rows_before - rows_after)],
    ["rows_before_dedupe", "rows_after_dedupe", "duplicates_removed"])
audit.write.mode("overwrite").saveAsTable("workspace.default.silver_audit")

s = spark.table("workspace.default.silver_life_satisfaction")
print("rows before:", rows_before, "| after:", rows_after)
print("missing values:", s.filter("is_missing").count())
print("flagged values (like 'u'):", s.filter("flag is not null").count())
print("out of range:", s.filter("is_out_of_range").count())
print("unreadable but not missing (should be 0):",
      s.filter("NOT is_missing AND value IS NULL").count())
display(s.groupBy("flag").count())

rows before: 43680 | after: 43680
missing values: 6756
flagged values (like 'u'): 11757
out of range: 0
unreadable but not missing (should be 0): 0


flag,count
null,31923
u,9156
m,498
b,984
e,627
bu,208
pu,23
p,261


In [0]:
from pyspark.sql import functions as F

s = spark.table("workspace.default.silver_life_satisfaction")
mystery = s.filter("NOT is_missing AND value IS NULL")

display(mystery.groupBy("raw_value").count().orderBy(F.desc("count")).limit(20))

raw_value,count


In [0]:
from pyspark.sql import functions as F

s = spark.table("workspace.default.silver_life_satisfaction")
bronze = spark.table("workspace.default.bronze_life_satisfaction")
total = s.count()

# ---- Gold 1: column health (one pass over the data) ----
cols = ["freq","statinfo","unit","isced11","life_sat","sex","age","geo","year","value"]
aggs = [F.sum(F.when(F.col(c).isNull() | (F.col(c).cast("string") == ""), 1).otherwise(0)).alias(c)
        for c in cols]
counts = s.agg(*aggs).first().asDict()
col_health = spark.createDataFrame(
    [(c, total, int(counts[c]), round(100 * counts[c] / total, 2)) for c in cols],
    ["column_name", "total_rows", "missing_count", "missing_pct"])
col_health.write.mode("overwrite").saveAsTable("workspace.default.gold_column_health")

# ---- Gold 2: overall scorecard ----
low_rel = s.filter(F.coalesce(F.col("flag").contains("u"), F.lit(False))).count()
bronze_rows = bronze.count()
dup_series = bronze_rows - bronze.select("series_key").distinct().count()

checks = [
    ("missing values (':' or ': letter')", s.filter("is_missing").count(), total),
    ("values carrying any quality flag", s.filter("flag IS NOT NULL").count(), total),
    ("low-reliability values (flag u)", low_rel, total),
    ("out of range (outside 0-10)", s.filter("is_out_of_range").count(), total),
    ("duplicate series in raw file", dup_series, bronze_rows),
]
scorecard = spark.createDataFrame(
    [(name, n, d, round(100 * n / d, 2)) for name, n, d in checks],
    ["check", "count", "out_of_rows", "pct"])
scorecard.write.mode("overwrite").saveAsTable("workspace.default.gold_quality_scorecard")

# ---- Gold 3: quality by country and by year ----
def quality_by(col):
    return (s.groupBy(col).agg(
        F.count("*").alias("rows"),
        F.round(100 * F.avg(F.col("is_missing").cast("int")), 1).alias("missing_pct"),
        F.round(100 * F.avg(F.coalesce(F.col("flag").contains("u"), F.lit(False)).cast("int")), 1)
            .alias("low_reliability_pct"))
        .orderBy(F.desc("missing_pct")))

by_geo = quality_by("geo")
by_year = quality_by("year")
by_geo.write.mode("overwrite").saveAsTable("workspace.default.gold_quality_by_country")
by_year.write.mode("overwrite").saveAsTable("workspace.default.gold_quality_by_year")

display(scorecard)
display(col_health)
display(by_geo.limit(10))
display(by_year)

check,count,out_of_rows,pct
missing values (':' or ': letter'),6756,43680,15.47
values carrying any quality flag,11757,43680,26.92
low-reliability values (flag u),9387,43680,21.49
out of range (outside 0-10),0,43680,0.0
duplicate series in raw file,0,6240,0.0


column_name,total_rows,missing_count,missing_pct
freq,43680,0,0.0
statinfo,43680,0,0.0
unit,43680,0,0.0
isced11,43680,0,0.0
life_sat,43680,0,0.0
sex,43680,0,0.0
age,43680,0,0.0
geo,43680,0,0.0
year,43680,0,0.0
value,43680,6756,15.47


geo,rows,missing_pct,low_reliability_pct
XK,1092,86.5,1.4
EA21,1092,86.2,0.5
IS,1092,74.1,5.1
UK,1092,72.5,15.3
EA20,1092,46.2,3.3
MK,1092,30.5,17.8
EU27_2020,1092,21.5,35.8
ME,1092,17.9,6.5
CH,1092,17.3,7.3
PT,1092,17.0,84.7


year,rows,missing_pct,low_reliability_pct
2021,6240,22.9,19.9
2025,6240,18.3,15.0
2013,6240,16.2,40.9
2023,6240,14.1,18.4
2022,6240,14.0,18.1
2024,6240,13.6,15.4
2018,6240,9.1,22.7


Databricks visualization. Run in Databricks to view.